# 7 · Leaderboard final y conclusiones

Cierre: reunimos todos los modelos en un **leaderboard** y sacamos las conclusiones.
Incluimos **DeepSVDD** como referencia de un método moderno de deep anomaly detection
(librería `deepod`), a presupuesto parejo (300 épocas, capacidad como el VAE).

> **Sobre los métodos modernos:** probamos también ICL y NeuTraL, pero con presupuesto
> parejo dieron **por debajo del baseline** (ICL) o **cerca del azar** (NeuTraL) — no aportan
> a la comparación, así que dejamos solo DeepSVDD (el mejor de los tres). A DeepSVDD **sí lo
> tuneamos** (7.0) para que la comparación contra el VAE sea justa: el VAE tuvo su búsqueda de
> HP (nb 3), así que le damos a DeepSVDD la suya.

In [1]:
import lab                      # utilidades: métricas y gráficos (experiments/lab.py)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data
from src.models import (IsolationForestDetector, OneClassSVMDetector,
                        ZScoreDetector, MahalanobisDetector,
                        AEDetector, DenoisingAEDetector, VAEDetector,
                        EnsembleDetector, DeepODDetector)
plt.rcParams["figure.dpi"] = 110
SEEDS = lab.SEEDS            # 10 semillas del estudio. Bajalas (p. ej. [42,43,44])
print("semillas:", SEEDS)   # para un Run all más rápido; los números se mueven ±std

semillas: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [2]:
panel_z = data.prepare()                         # panel + etiqueta z_rinde
ds   = data.build_crop_dataset(panel_z, "soja")  # split + normalización (soja)
dsm  = data.build_crop_dataset(panel_z, "maiz")  # idem maíz
print("soja  train:", ds.X_train.shape, "| test anómalas:", int(ds.y_test.sum()))
print("maíz  train:", dsm.X_train.shape, "| test anómalas:", int(dsm.y_test.sum()))

[data] dedup panel: 27862 -> 20755 filas (7107 duplicados espurios por lat/lon eliminados)


soja  train: (6143, 65) | test anómalas: 251
maíz  train: (8082, 65) | test anómalas: 344


In [ ]:
datasets = {"soja": ds, "maiz": dsm}
BEST = {
    "soja": dict(hidden_dims=(128, 64), latent_dim=16),
    "maiz": dict(hidden_dims=(128, 64), latent_dim=24),
}  # config ganadora del VAE por cultivo (nb 3) — no es la misma arquitectura para los dos

## 7.0 · Tuning de DeepSVDD (comparación justa, por cultivo)
Barrido chico de `lr`, `rep_dim` y `hidden_dims`, seleccionado **multi-seed** por
`media − std` (mismo criterio que el VAE en el nb 3: premia lo bueno *y* estable), para
**soja y maíz por separado**. El resto (`epochs=300`, `batch_size=64`) queda a presupuesto
parejo con el VAE.

In [ ]:
DSVDD_GRID = [dict(lr=lr, rep_dim=rd, hidden_dims=hd)
              for lr in (1e-3, 5e-4) for rd in (16, 24) for hd in ("64,32", "128,64")]

DSVDD_BEST = {}
dsvdd_hp_todos = {}
for cultivo, dset in datasets.items():
    filas_dsvdd = []
    for cfg in DSVDD_GRID:
        prs = [lab.metrics(DeepODDetector(algo="deepsvdd", epochs=300, batch_size=64,
                           random_state=s, **cfg).fit(dset.X_train).score_samples(dset.X_test),
                           dset.y_test)["pr_auc"] for s in SEEDS[:3]]
        filas_dsvdd.append({**cfg, "pr_auc_mean": round(np.mean(prs), 3),
                            "pr_auc_std": round(np.std(prs), 3),
                            "media_menos_std": round(np.mean(prs) - np.std(prs), 3)})
    dsvdd_hp = pd.DataFrame(filas_dsvdd).sort_values("media_menos_std", ascending=False).reset_index(drop=True)
    dsvdd_hp_todos[cultivo] = dsvdd_hp
    DSVDD_BEST[cultivo] = dict(lr=float(dsvdd_hp.iloc[0]["lr"]), rep_dim=int(dsvdd_hp.iloc[0]["rep_dim"]),
                              hidden_dims=dsvdd_hp.iloc[0]["hidden_dims"])
    print(f"DeepSVDD tuneado ({cultivo}, mejor media−std):", DSVDD_BEST[cultivo])
    display(dsvdd_hp)

## 7.1 · Leaderboard final (soja y maíz)
Entrenamos cada modelo clave acá mismo, sobre el **panel unificado** (clima + NDVI + ERA5).
El **modelo final** es el seed-ensemble ×10 del VAE `recon_prob`, con la config ganadora
**de cada cultivo** (nb 3: `latent_dim=16` en soja, `24` en maíz). Todo multi-seed; OCSVM es
determinista (std 0).

In [ ]:
def ens_scores(cultivo, dset, base):
    miembros = [VAEDetector(**BEST[cultivo], score_mode="recon_prob",
                n_mc_samples=50, max_epochs=300, patience=30, random_state=base+i) for i in range(10)]
    return EnsembleDetector(miembros).fit(dset.X_train).score_samples(dset.X_test)

lb = {}
for cultivo, dset in datasets.items():
    lb[cultivo] = lab.leaderboard({
      "VAE seed-ensemble ★": [lab.metrics(ens_scores(cultivo, dset, b), dset.y_test) for b in [42,52,62]],
      "VAE recon_prob single": [lab.metrics(VAEDetector(**BEST[cultivo],
            score_mode="recon_prob", n_mc_samples=50, max_epochs=300, patience=30, random_state=s)
            .fit(dset.X_train).score_samples(dset.X_test), dset.y_test) for s in SEEDS],
      "AE (MSE)": [lab.metrics(AEDetector(hidden_dims=(128,64), latent_dim=24, score_mode="mse",
            max_epochs=300, patience=30, random_state=s)
            .fit(dset.X_train).score_samples(dset.X_test), dset.y_test) for s in SEEDS],
      "DAE (MSE)": [lab.metrics(DenoisingAEDetector(hidden_dims=(128,64), latent_dim=24, score_mode="mse",
            max_epochs=300, patience=30, random_state=s)
            .fit(dset.X_train).score_samples(dset.X_test), dset.y_test) for s in SEEDS],
      "IForest": [lab.metrics(IsolationForestDetector(n_estimators=200, max_features=0.3, random_state=s)
            .fit(dset.X_train).score_samples(dset.X_test), dset.y_test) for s in SEEDS],   # estocástico → std
      "DeepSVDD (moderno, tuneado)": [lab.metrics(DeepODDetector(algo="deepsvdd", epochs=300,
            batch_size=64, random_state=s, **DSVDD_BEST[cultivo])
            .fit(dset.X_train).score_samples(dset.X_test), dset.y_test) for s in SEEDS],
      "OCSVM-RBF (determinista)": [lab.metrics(OneClassSVMDetector(kernel="rbf", nu=0.1)
            .fit(dset.X_train).score_samples(dset.X_test), dset.y_test)],                   # 1 corrida → std 0
    })
    print(f"--- {cultivo} ---")
    lab.plot_leaderboard(lb[cultivo]); plt.show()
    display(lb[cultivo][["modelo","pr_auc","pr_auc_std","roc_auc"]].round(3))

## 7.2 · Recapitulación del recorrido (soja y maíz)
La historia en una tabla. Los **PR-AUC salen del leaderboard vivo de arriba** (`lb`, celda
7.1) — no hay ningún número escrito a mano: si cambiás las semillas o el pipeline, esta tabla
se mueve sola con la corrida.

In [ ]:
etapas = [
    ("nb1", "Isolation Forest (baseline)",  "IForest",                       "listón no neuronal, varianza baja"),
    ("nb1", "One-Class SVM (RBF)",          "OCSVM-RBF (determinista)",     "kernel gaussiano ≈ IForest; el lineal fracasa (nb1)"),
    ("nb2", "AE (score MSE)",               "AE (MSE)",                      "el MSE plano no alcanza"),
    ("nb2", "DAE (score MSE)",              "DAE (MSE)",                     "el denoising no cambia el cuello de botella"),
    ("nb3", "VAE recon_prob (single)",      "VAE recon_prob single",         "el SCORE probabilístico es el salto vs MSE"),
    ("nb4", "VAE seed-ensemble ×10 ★",      "VAE seed-ensemble ★",           "MODELO FINAL: baja la varianza y sube la media"),
    ("nb7", "DeepSVDD (moderno, tuneado)",  "DeepSVDD (moderno, tuneado)",   "tuneado (7.0) y a presupuesto parejo, por debajo del VAE"),
]

filas = []
for cultivo in datasets:
    pr = dict(zip(lb[cultivo]["modelo"], lb[cultivo]["pr_auc"]))   # PR-AUC reales, tomados del leaderboard 7.1
    for nb_, etapa, clave, conclusion in etapas:
        filas.append({"cultivo": cultivo, "nb": nb_, "etapa": etapa,
                      "PR-AUC (test)": round(pr[clave], 3), "conclusión": conclusion})
recap = pd.DataFrame(filas)
recap

## Conclusiones del Componente A
1. **Modelo final: seed-ensemble ×10 del VAE `recon_prob`** sobre el panel unificado (clima
   + NDVI + ERA5) — el mejor del leaderboard de arriba (7.1) y con el menor desvío, **en
   soja y en maíz**. Ojo: la arquitectura ganadora no es la misma para los dos cultivos
   (`latent_dim=16` en soja, `24` en maíz, nb 3) — un modelo por cultivo, no uno compartido.
2. **Dos decisiones** lo explican: el **score probabilístico** `recon_prob` (aplasta al MSE,
   nb 3) y el **ensemble de semillas** que baja fuerte la varianza (nb 4). El panel ya trae
   suelo + verdor (NDVI/ERA5), que corren el techo un poco (mejora chica pero real; por eso
   quedaron en el dataset).
3. **Lección metodológica central:** reportar la varianza multi-seed no es un detalle — sin
   ella, mejoras chicas (como la de suelo+verdor) quedan enterradas en el ruido de los
   modelos single.
4. Alternativas refutadas: AE/DAE con MSE, detectores sobre el latente (nb 2–4), las features
   `agro` de dominio (nb 6), y el deep AD moderno tuneado a presupuesto parejo (nb 7) — todo
   consistente entre cultivos.
5. **Techo estructural (nb 5):** existe (rondamos ~0.6 en soja, algo más bajo en maíz), la
   mayoría de las anomalías siguen sin firma observable. Superarlo más requeriría **feature
   engineering más rico** (nb 6) u **otra clase de datos** (sanidad, granizo, manejo).